What it is: Self-Reflection Memory has the agent analyze its own performance after tasks, extract lessons learned, and store them for future retrieval.

When you need it: Your agent repeats similar tasks and you want it to improve by learning from past mistakes and successes.

The trade-off: Reflection quality depends on model reasoning ability, and every task triggers an extra LLM call for analysis.

Closest alternative in this repo: Episodic Memory records what happened, while Self-Reflection Memory records what the agent learned from it.

-----------------------------------------------------------------------------------------
After each task attempt, the agent pauses to reflect on what happened, extracts durable insights, and stores them in a reflection memory. This way, the same mistakes never repeat and successful strategies carry forward across sessions.

Think of a chess player who reviews their games after each match. They don't only record the moves. They ask: "Why did I lose that piece? What pattern did I miss?" Over time, those post-game notes become more valuable than the game records themselves.

Most memory techniques record what happened in a conversation: messages, entities, summaries. Self-Reflection Memory takes a different approach. It records why things went well or poorly, and what to do differently next time. This is the core idea behind Reflexion (Shinn et al., 2023), where agents use verbal self-reflection to convert sparse outcome signals into rich, reusable learning.

The pattern works in four steps:

The agent attempts a task and receives an outcome (success, partial, failure).

A reflection prompt asks the agent to analyze root causes and extract lessons.

The resulting insight goes into a dedicated reflection memory.

Before future tasks, the agent retrieves relevant past reflections and injects them into the prompt.

This creates a reflect-then-store loop. The loop lets an LLM-based agent (a language model used as a reasoning engine) improve across attempts without any parameter updates. Learning lives entirely in natural-language reflections.

### Key Concepts

Reflection prompt: A structured set of questions that guides the agent to evaluate its own performance. Example: "What went well? What failed? What's the root cause? What should I do differently?" This converts a pass/fail outcome into a rich learning signal.

Insight extraction: Distilling a full reflection into a concise, reusable principle. For example: "Always check for edge cases in date parsing before attempting conversion." Insights are the unit of storage.

Reflection memory store: A dedicated memory that holds past reflections. They're indexed by task type and outcome. Before each new task, the agent retrieves relevant reflections and injects them into its context.

Reflect-then-store loop: The core pattern: attempt, evaluate outcome, reflect, extract insight, store, retrieve on next attempt. Each cycle adds to the agent's accumulated wisdom.

Outcome evaluation: Comparing the agent's output against expected results to determine success, partial success, or failure. This signal triggers reflection.

Reflexion framework: From Shinn et al. (2023). Agents use verbal reinforcement learning (learning through written self-feedback). Natural-language reflections replace gradient updates as the learning mechanism.

Meta-cognition: The agent's ability to reason about its own reasoning. Self-reflection memory is an explicit implementation of metacognitive monitoring (watching your own thought process).

## What Is Self-Reflection Memory?

### The core idea in one sentence
After each session, the agent reviews its own responses — identifying what it did well, what it did poorly, and what it should do differently — and writes structured improvement notes that are injected into future sessions.

---

### The mental model — a doctor's debrief

After a difficult consultation, a thoughtful doctor does not just move on to the next patient. They spend five minutes reviewing: Did I ask the right questions? Did I miss any symptoms? Was my communication clear? Would I diagnose differently in hindsight?

Those notes go in a personal learning journal. The next time a similar patient presents, the doctor reads those notes first. Their practice improves — not through formal training, but through structured self-critique.

Self-reflection memory gives FinCoach the same capability: structured post-session critique that feeds forward into every subsequent interaction.

---

### How it differs from Procedural Memory (Technique 11)

Both techniques improve the agent over time. The difference is the source:

| | Procedural (11) | Self-Reflection (16) |
|---|---|---|
| **Source** | Patterns extracted from session content | Agent's own critique of its performance |
| **Perspective** | Third-party observation | First-person self-assessment |
| **Content** | "How to handle X" | "I should have done Y instead of Z" |
| **Mechanism** | Extraction from transcript | Deliberate reflection post-session |
| **Injects as** | System prompt directives | Self-awareness notes |
| **Papers** | Voyager, Agent Workflow Memory | Reflexion, ExpeL, Hindsight is 20/20 |

Procedural memory learns what to do. Self-reflection learns from mistakes.

---

### The Reflexion paper — the research foundation

Systems such as Reflexion and ExpeL allow agents to write verbal post-mortems and store conclusions for future runs. Reflexion (Shinn et al., 2023) demonstrated that agents given the ability to reflect on their own failed attempts — and incorporate those reflections into subsequent attempts — significantly outperformed agents that simply tried again without reflection. The key mechanism: verbal reinforcement through self-written feedback, not parameter updates.

---

### Five dimensions of self-reflection for FinCoach

**1. Consistency check**: Did the advice I gave contradict what I said in previous sessions?

**2. Completeness check**: Did I miss any important financial angle the user should have considered?

**3. Constraint compliance**: Did I respect the user's stated constraints ("never equity")? Any violations?

**4. Communication quality**: Was I clear? Did I use the right level of detail? Did I provide the action list the user prefers?

**5. Missed signals**: Were there emotional cues (anxiety, frustration) I could have handled better?

Each dimension generates a structured reflection note. High-severity notes are injected on every subsequent call. Low-severity notes are stored and surfaced only when similar situations arise.

---

### The production failure mode: hallucinated self-critique

The idea is compelling — agents learn from mistakes and improve. The failure mode is severe. The risk is the agent generating false self-critiques — claiming it made mistakes it did not make, or praising itself for non-existent successes. This pollutes the reflection store with inaccurate feedback that degrades performance over time.

The mitigation: ground every reflection in a specific, quoted excerpt from the session. A reflection without evidence is flagged as low-confidence. Only evidence-backed reflections are injected.

---

## Trade-offs

| | |
|---|---|
| ✅ Agent learns from mistakes without retraining | ❌ Risk of hallucinated self-critique |
| ✅ Catches systematic errors before they compound | ❌ Reflection quality depends on prompt engineering |
| ✅ Builds agent self-awareness over time | ❌ Extra LLM call per session (post-session, async) |
| ✅ Surfaces constraint violations for audit | ❌ Reflection can become stale — old mistakes persist |
| ✅ Combines naturally with procedural memory | ❌ Requires outcome signal to prioritise what to reflect on |

## Production Verdict

> **High-value pattern for regulated domains. The audit trail it produces is worth the extra LLM call.**
>
> In financial services, self-reflection serves two purposes simultaneously: agent quality improvement (learning from mistakes) and compliance (detecting and logging when the agent violated constraints or gave inconsistent advice). The reflection store is both a learning mechanism and an audit trail. Build it for any domain where advice quality and consistency matter.

---

In [1]:
!pip install openai tiktoken --quiet

In [2]:
import json, os, time, hashlib
from datetime import datetime, timezone
from typing import List, Dict, Optional, Any
from dataclasses import dataclass, field, asdict
from enum import Enum

import openai
import tiktoken

In [3]:
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    print("Key loaded from Colab Secrets.")
except Exception:
    OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
    print("Key loaded from environment variable.")

assert OPENAI_API_KEY and OPENAI_API_KEY.startswith("sk-"), "API key missing."

client          = openai.OpenAI(api_key=OPENAI_API_KEY)
CHAT_MODEL      = "gpt-4o"
REFLECTION_MODEL = "gpt-4o-mini"
# Reflection is a structured critique task — mini is sufficient.
# Using mini keeps the reflection cost low (~$0.0001 per session).
TOKENISER       = tiktoken.get_encoding("o200k_base")

print(f"Chat model      : {CHAT_MODEL}")
print(f"Reflection model: {REFLECTION_MODEL}")

Key loaded from environment variable.
Chat model      : gpt-4o
Reflection model: gpt-4o-mini


---
## Part 1 — The Reflection Data Model

In [4]:
class ReflectionDimension(str, Enum):
    """The five dimensions the agent critiques itself on."""
    CONSISTENCY   = "consistency"    # Did advice contradict previous sessions?
    COMPLETENESS  = "completeness"   # Did I cover all important angles?
    COMPLIANCE    = "compliance"     # Did I respect stated user constraints?
    COMMUNICATION = "communication"  # Was delivery clear and appropriate?
    MISSED_SIGNAL = "missed_signal"  # Did I miss emotional or contextual cues?


class ReflectionSeverity(str, Enum):
    """How urgent is this reflection?"""
    CRITICAL = "critical"   # Constraint violated — always inject, flag for audit.
    HIGH     = "high"       # Clear mistake — inject until resolved.
    MEDIUM   = "medium"     # Worth noting — inject for similar situations.
    LOW      = "low"        # Minor improvement — store but rarely inject.
    POSITIVE = "positive"   # What went well — reinforce.


@dataclass
class Reflection:
    """
    A single self-reflection note generated after a session.
    Captures what the agent did (well or poorly) and what should change.
    """

    reflection_id:  str
    # Hash of (user_id + dimension + observation[:60]) for deduplication.

    user_id:        str

    session_id:     str
    # Which session generated this reflection.

    dimension:      str
    # One of ReflectionDimension values.

    severity:       str
    # One of ReflectionSeverity values.

    observation:    str
    # What the agent observed about its own behaviour.
    # Example: "I gave investment advice without confirming risk profile first."

    evidence:       str
    # Quoted excerpt from the session that supports this observation.
    # Grounds the reflection — prevents hallucinated self-critique.
    # Example: "User said 'I'm not sure about risk' but I proceeded to recommend equity."

    improvement:    str
    # What the agent should do differently next time.
    # Example: "Always confirm risk tolerance explicitly before recommending products."

    resolved:       bool
    # Has this issue been addressed in a subsequent session?
    # Resolved reflections are archived and no longer injected.

    recurrence_count: int
    # How many sessions has this pattern appeared?
    # High recurrence = systematic issue, not one-off.

    created_at:     str
    last_seen:      str

    def should_inject(self, min_severity: str = ReflectionSeverity.MEDIUM) -> bool:
        """
        Whether this reflection should be injected into the next session.
        Always inject: CRITICAL (compliance violations)
        Inject if not resolved: HIGH, MEDIUM (above threshold)
        Rarely inject: LOW
        """
        if self.resolved:
            return False
            # Resolved reflections are no longer injected.

        severity_order = {
            ReflectionSeverity.CRITICAL: 4,
            ReflectionSeverity.HIGH:     3,
            ReflectionSeverity.MEDIUM:   2,
            ReflectionSeverity.LOW:      1,
            ReflectionSeverity.POSITIVE: 0,
        }
        return severity_order.get(self.severity, 0) >= severity_order.get(min_severity, 2)

    def format_for_injection(self) -> str:
        """Format as a self-awareness note for system prompt injection."""
        severity_icon = {
            ReflectionSeverity.CRITICAL: "🚨",
            ReflectionSeverity.HIGH:     "⚠",
            ReflectionSeverity.MEDIUM:   "📝",
            ReflectionSeverity.LOW:      "💡",
            ReflectionSeverity.POSITIVE: "✅",
        }.get(self.severity, "📝")

        lines = [
            f"{severity_icon} [{self.dimension.upper()}] Reflection from session {self.session_id}:",
            f"  Observed  : {self.observation}",
            f"  Improve   : {self.improvement}",
        ]
        if self.recurrence_count > 1:
            lines.append(f"  Recurrence: {self.recurrence_count} sessions — systematic pattern")
        return "\n".join(lines)

    def mark_resolved(self) -> None:
        """Mark this reflection as addressed — stops injection."""
        self.resolved = True

    def record_recurrence(self, session_id: str) -> None:
        """Record that this pattern appeared again in a new session."""
        self.recurrence_count += 1
        self.last_seen = datetime.now(timezone.utc).isoformat()


print("Reflection data model defined.")

Reflection data model defined.


---
## Part 2 — The Reflection Generator

The reflection generator reads a session transcript and produces structured self-critique notes. This is the agent examining its own work.

In [5]:
REFLECTION_PROMPT = """You are a quality assurance reviewer for FinCoach, an AI financial advisor.
Your job is to critically review the agent's performance in a session and identify
what it did well and what it should improve.

Review the session transcript on these five dimensions:
1. CONSISTENCY    — Did advice contradict previous sessions or stated user facts?
2. COMPLETENESS   — Did the agent miss important financial considerations?
3. COMPLIANCE     — Were user constraints respected? (e.g. 'never equity', 'no crypto')
4. COMMUNICATION  — Was the response clear, appropriately detailed, well-structured?
5. MISSED_SIGNAL  — Were emotional or contextual cues handled correctly?

Return a JSON object:
{
  "reflections": [
    {
      "dimension"   : "consistency | completeness | compliance | communication | missed_signal",
      "severity"    : "critical | high | medium | low | positive",
      "observation" : "What the agent did (well or poorly) — specific and factual",
      "evidence"    : "Exact quote or paraphrase from the session that shows this",
      "improvement" : "What the agent should do differently next time"
    }
  ]
}

SEVERITY GUIDE:
- critical : constraint violated OR harmful advice given — must flag for audit
- high     : clear mistake that affected advice quality
- medium   : noticeable gap worth correcting
- low      : minor improvement opportunity
- positive : something done well — reinforce it

STRICT RULES:
1. Return ONLY valid JSON — no markdown, no explanation.
2. Every reflection MUST have evidence — no evidence means no reflection.
   This prevents hallucinated self-critique.
3. Be specific: cite what was actually said, not hypothetical mistakes.
4. Maximum 5 reflections — prioritise by severity.
5. If the session had no notable issues, return 1-2 positive reflections only.
6. Compliance violations (severity=critical) must always be included if present.
"""
# The 'evidence required' rule is the key production safeguard.
# It grounds every reflection in the actual session, preventing fabrication.


def generate_reflections(
    session_id: str,
    user_id: str,
    transcript: str,
    user_context: str = "",
) -> List[Reflection]:
    """
    Generate self-reflection notes from a session transcript.

    Args:
        session_id:    The session being reviewed.
        user_id:       The user's ID.
        transcript:    Full session transcript as a string.
        user_context:  Optional: known user facts to check compliance against.
                       Example: "User stated they never want equity investments."

    Returns:
        List of Reflection objects ready for merging into the store.
    """

    content = transcript
    if user_context:
        content = f"USER CONTEXT (known facts/constraints):\n{user_context}\n\nSESSION TRANSCRIPT:\n{transcript}"
        # Providing user context allows the reviewer to check compliance
        # against stated constraints — the most critical check.

    response = client.chat.completions.create(
        model=REFLECTION_MODEL,
        max_tokens=800,
        temperature=0.0,
        # Deterministic — critique should not vary on the same session.
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": REFLECTION_PROMPT},
            {"role": "user",   "content":
             f"Review this FinCoach session and generate reflections:\n\n{content}"},
        ]
    )

    raw = json.loads(response.choices[0].message.content)
    raw_refs = raw.get("reflections", [])

    reflections = []
    now = datetime.now(timezone.utc).isoformat()

    for rr in raw_refs:
        observation = rr.get("observation", "").strip()
        evidence    = rr.get("evidence", "").strip()

        if not observation or not evidence:
            # Skip reflections without both observation AND evidence.
            # Evidence is mandatory — no grounding = no injection.
            continue

        # Stable ID: same observation for same user → same ID → deduplication.
        ref_id = hashlib.md5(
            f"{user_id}:{rr.get('dimension','')[:20]}:{observation[:60]}".encode()
        ).hexdigest()[:12]

        ref = Reflection(
            reflection_id   = ref_id,
            user_id         = user_id,
            session_id      = session_id,
            dimension       = rr.get("dimension",    ReflectionDimension.COMMUNICATION),
            severity        = rr.get("severity",     ReflectionSeverity.MEDIUM),
            observation     = observation,
            evidence        = evidence,
            improvement     = rr.get("improvement",  "").strip(),
            resolved        = False,
            recurrence_count = 1,
            created_at      = now,
            last_seen       = now,
        )
        reflections.append(ref)

    tokens_used = response.usage.total_tokens
    print(f"  [REFLECT] {len(reflections)} reflections generated | tokens: {tokens_used}")
    for r in reflections:
        print(f"    [{r.severity.upper():8}] [{r.dimension:15}] {r.observation[:60]}...")

    return reflections


print("Reflection generator defined.")

Reflection generator defined.


---
## Part 3 — The ReflectionStore

In [6]:
class ReflectionStore:
    """
    Stores and manages self-reflection notes.
    Handles deduplication, recurrence tracking, severity filtering,
    and formatted injection into the system prompt or context block.
    """

    def __init__(
        self,
        user_id: str,
        min_severity_to_inject: str = ReflectionSeverity.MEDIUM,
        # Only inject reflections at or above this severity.
        max_reflections_in_context: int = 4,
        # Cap on how many reflections to inject per call.
        # Prioritises by severity then recurrence.
    ):
        self.user_id                  = user_id
        self.min_severity_to_inject   = min_severity_to_inject
        self.max_reflections_in_context = max_reflections_in_context

        self._reflections: Dict[str, Reflection] = {}
        # Key: reflection_id, Value: Reflection.

        self._reflection_count = 0
        # Total reflections ever generated — for analytics.

        self.created_at = datetime.now(timezone.utc).isoformat()

        print(f"[ReflectionStore] Initialised for user: {self.user_id}")

    def merge_reflections(self, new_refs: List[Reflection]) -> Dict[str, int]:
        """
        Merge newly generated reflections into the store.
        - Same reflection_id: record recurrence (same mistake repeated).
        - New reflection_id: insert as new.
        """
        added      = 0
        recurred   = 0

        for ref in new_refs:
            if ref.reflection_id in self._reflections:
                # Same observation again — this pattern is recurring.
                self._reflections[ref.reflection_id].record_recurrence(ref.session_id)
                recurred += 1
            else:
                self._reflections[ref.reflection_id] = ref
                added += 1

        self._reflection_count += len(new_refs)
        print(f"  [MERGE] +{added} new reflections, {recurred} recurrences")
        return {"added": added, "recurred": recurred}

    def get_active_reflections(self) -> List[Reflection]:
        """
        Return reflections that should be injected.
        Sorted: CRITICAL first, then by recurrence (most repeated = most important).
        """
        severity_order = {
            ReflectionSeverity.CRITICAL: 4,
            ReflectionSeverity.HIGH:     3,
            ReflectionSeverity.MEDIUM:   2,
            ReflectionSeverity.LOW:      1,
            ReflectionSeverity.POSITIVE: 0,
        }

        active = [
            r for r in self._reflections.values()
            if r.should_inject(self.min_severity_to_inject)
        ]

        active.sort(
            key=lambda r: (severity_order.get(r.severity, 0), r.recurrence_count),
            reverse=True
        )

        return active[:self.max_reflections_in_context]

    def get_compliance_violations(self) -> List[Reflection]:
        """Return all CRITICAL reflections — for compliance audit."""
        return [
            r for r in self._reflections.values()
            if r.severity == ReflectionSeverity.CRITICAL
        ]

    def format_for_context(self) -> str:
        """
        Format active reflections as a self-awareness block for the API call.
        This is injected as a system message — the agent is told about its own past behaviour.

        Design choice: injected as a context block (NOT merged into the system prompt).
        Rationale: reflections are situational — they belong as context, not as permanent
        operational rules. This separates them from Procedural Memory (Technique 11).
        """
        active = self.get_active_reflections()

        if not active:
            return ""

        lines = [
            "SELF-REFLECTION NOTES (review your past performance before responding):",
            "These are areas where your advice could improve based on previous sessions.",
            "",
        ]

        for ref in active:
            lines.append(ref.format_for_injection())
            lines.append("")

        lines.append(
            "Apply these improvements in your current response. "
            "Do not mention these notes to the user."
        )
        return "\n".join(lines)

    def token_count(self) -> int:
        """Token cost of injecting the reflection context block."""
        text = self.format_for_context()
        return len(TOKENISER.encode(text)) if text else 0

    def mark_resolved(self, reflection_id: str) -> None:
        """Mark a specific reflection as resolved — stops injection."""
        if reflection_id in self._reflections:
            self._reflections[reflection_id].mark_resolved()
            print(f"  [RESOLVED] Reflection {reflection_id} marked resolved.")

    def resolve_dimension(self, dimension: str) -> int:
        """Resolve all reflections of a given dimension (e.g. after a pattern is fixed)."""
        count = 0
        for ref in self._reflections.values():
            if ref.dimension == dimension and not ref.resolved:
                ref.mark_resolved()
                count += 1
        print(f"  [RESOLVED] {count} reflections resolved for dimension: {dimension}")
        return count

    def persist(self, filepath: str) -> None:
        """Save all reflections to JSON."""
        record = {
            "schema_version":     "1.0",
            "technique":          "self_reflection_memory",
            "user_id":            self.user_id,
            "created_at":         self.created_at,
            "saved_at":           datetime.now(timezone.utc).isoformat(),
            "stats": {
                "total_reflections":  self._reflection_count,
                "active_count":       len(self.get_active_reflections()),
                "compliance_count":   len(self.get_compliance_violations()),
                "context_tokens":     self.token_count(),
            },
            "reflections": {
                rid: asdict(ref)
                for rid, ref in self._reflections.items()
            },
        }
        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(record, f, indent=2, ensure_ascii=False)
        print(f"[ReflectionStore] Persisted — {len(self._reflections)} total, "
              f"{len(self.get_active_reflections())} active")

    @classmethod
    def load(cls, filepath: str) -> "ReflectionStore":
        """Restore ReflectionStore from JSON."""
        with open(filepath, "r", encoding="utf-8") as f:
            record = json.load(f)
        store = cls(user_id=record["user_id"])
        for rid, rdict in record["reflections"].items():
            store._reflections[rid] = Reflection(**rdict)
        store._reflection_count = record["stats"]["total_reflections"]
        store.created_at = record["created_at"]
        print(f"[ReflectionStore] Loaded — {len(store._reflections)} reflections")
        return store

    def print_all_reflections(self) -> None:
        """Print all reflections with full metadata."""
        print(f"\n{'='*65}")
        print(f"  Reflection Store — User: {self.user_id}")
        print(f"  Total: {len(self._reflections)} | Active: {len(self.get_active_reflections())} | "
              f"Violations: {len(self.get_compliance_violations())} | "
              f"Tokens: ~{self.token_count()}")
        print(f"{'='*65}")
        for rid, ref in sorted(
            self._reflections.items(),
            key=lambda x: x[1].recurrence_count, reverse=True
        ):
            status = "[ACTIVE  ]" if ref.should_inject(self.min_severity_to_inject) else "[RESOLVED]"
            print(f"\n  {status} [{ref.severity.upper():8}] [{ref.dimension}] "
                  f"recurred: {ref.recurrence_count}x")
            print(f"  Observed  : {ref.observation[:80]}...")
            print(f"  Evidence  : {ref.evidence[:70]}...")
            print(f"  Improve   : {ref.improvement[:80]}...")
        print(f"{'='*65}\n")


print("ReflectionStore defined.")

ReflectionStore defined.


---
## Part 4 — The SelfReflectionMemory Class

In [7]:
class SelfReflectionMemory:
    """
    Full self-reflection memory system for FinCoach.

    Manages:
    - ReflectionStore: all past self-critique notes
    - Current session buffer: for conversation flow
    - Post-session reflection: triggered at session close
    - Context injection: reflections appear as self-awareness notes

    The agent is shown its own past errors and good practices before each turn.
    This is NOT shown to the user — it is internal agent self-awareness.
    """

    def __init__(
        self,
        session_id: str,
        user_id: str,
        system_prompt: str,
        user_context: str = "",
        # Known user facts/constraints — used for compliance checking.
        max_recent_turns: int = 5,
        min_severity_to_inject: str = ReflectionSeverity.MEDIUM,
    ):
        self.session_id     = session_id
        self.user_id        = user_id
        self.system_prompt  = system_prompt
        self.user_context   = user_context
        self.max_recent_turns = max_recent_turns

        self.reflection_store = ReflectionStore(
            user_id=user_id,
            min_severity_to_inject=min_severity_to_inject,
        )

        self._recent_buffer: List[Dict] = []
        self._full_session:  List[Dict] = []
        self._total_turns = 0
        self.created_at   = datetime.now(timezone.utc).isoformat()

        print(f"[SelfReflection] Initialised — session: {self.session_id}")
        print(f"  Active reflections: {len(self.reflection_store.get_active_reflections())}")

    def add_message(self, role: str, content: str) -> None:
        """Add to buffer and full session log."""
        if role not in ("user", "assistant"):
            raise ValueError(f"Invalid role '{role}'.")
        msg = {"role": role, "content": content}
        self._recent_buffer.append(msg)
        if len(self._recent_buffer) > self.max_recent_turns * 2:
            self._recent_buffer.pop(0)
        self._full_session.append(msg)
        if role == "assistant":
            self._total_turns += 1

    def get_messages_for_api(self) -> List[Dict[str, str]]:
        """
        Build the API message list.

        Structure:
          [system: FinCoach persona]
          [system: self-reflection notes]  ← agent's self-awareness
          [recent buffer]

        The reflection notes are injected as a second system message.
        The agent reads them before generating a response — self-awareness
        shapes the reply without the user seeing the notes.
        """
        messages = []
        messages.append({"role": "system", "content": self.system_prompt})

        reflection_block = self.reflection_store.format_for_context()
        if reflection_block:
            messages.append({"role": "system", "content": reflection_block})
            # Agent reviews its own past notes before each response.

        for msg in self._recent_buffer:
            messages.append(msg)

        return messages

    def reflect_on_session(self) -> List[Reflection]:
        """
        Generate self-reflection notes from the current session.
        Call at session close — runs asynchronously in production.
        """
        if not self._full_session:
            print("  [REFLECT] No messages to reflect on.")
            return []

        transcript = "\n".join(
            f"{m['role'].upper()}: {m['content']}"
            for m in self._full_session
        )

        print(f"\n[SelfReflection] Reflecting on session {self.session_id}...")

        new_refs = generate_reflections(
            session_id   = self.session_id,
            user_id      = self.user_id,
            transcript   = transcript,
            user_context = self.user_context,
        )

        if new_refs:
            self.reflection_store.merge_reflections(new_refs)

        return new_refs

    def print_stats(self) -> None:
        active = self.reflection_store.get_active_reflections()
        violations = self.reflection_store.get_compliance_violations()
        print(f"\n{'='*58}")
        print(f"  Self-Reflection Memory — Session: {self.session_id}")
        print(f"{'='*58}")
        print(f"  Session turns        : {self._total_turns}")
        print(f"  Active reflections   : {len(active)}")
        print(f"  Compliance violations: {len(violations)}")
        print(f"  Reflection tokens    : ~{self.reflection_store.token_count()}")
        print(f"{'='*58}")
        self.reflection_store.print_all_reflections()


print("SelfReflectionMemory defined.")

SelfReflectionMemory defined.


---
## Part 5 — The FinCoach Agent

In [8]:
FINCOACH_SYSTEM_PROMPT = """You are FinCoach, a personal financial advisor assistant.
You serve users in India who want guidance on savings, investments, budgeting, and financial planning.

Core principles:
- Always confirm risk tolerance before making investment recommendations.
- Personalise advice using information shared in the conversation.
- Never recommend specific stocks.
- Always recommend consulting a SEBI-registered advisor for major decisions.
- If self-reflection notes are provided, apply them silently — do not mention them."""
# The last line is important: the model should apply the reflections without
# saying "I noticed from my notes that..." — it should just do better.

# User context for compliance checking during reflection.
CHIRU_USER_CONTEXT = """Known facts about this user:
- Name: Chiru
- Monthly salary: ₹1,20,000
- Monthly expenses: ₹60,000
- Risk profile: CONSERVATIVE
- Hard constraint: NEVER recommend equity or stock market investments
- Prefers: action lists at end of sessions, worst-case scenarios upfront
- Goal: retire at 55 (currently 32)"""


def chat(
    memory: SelfReflectionMemory,
    user_message: str,
    verbose: bool = True
) -> str:
    """Send one message to FinCoach with self-reflection memory."""
    memory.add_message(role="user", content=user_message)

    response = client.chat.completions.create(
        model=CHAT_MODEL,
        max_tokens=1024,
        temperature=0.7,
        messages=memory.get_messages_for_api(),
        # System prompt + self-reflection notes + recent buffer.
    )

    reply = response.choices[0].message.content
    memory.add_message(role="assistant", content=reply)

    if verbose:
        active_count = len(memory.reflection_store.get_active_reflections())
        ref_tokens   = memory.reflection_store.token_count()
        print(f"[Turn {memory._total_turns}] "
              f"API: {response.usage.prompt_tokens} tokens | "
              f"Active reflections: {active_count} ({ref_tokens} tokens)")

    return reply


print("FinCoach chat() with self-reflection memory defined.")

FinCoach chat() with self-reflection memory defined.


---
## Part 6 — Demo: Session 1 — The Mistake Session

Session 1 contains deliberate quality issues — equity recommendation to a conservative user, missing action list, unclear structure. After the session, reflection generates critical notes.

In [9]:
# Session 1: FinCoach makes a compliance error — recommends equity
# to a user who has a hard constraint against it.
# We inject a flawed assistant turn to force the reflection to find it.

ref_mem_s1 = SelfReflectionMemory(
    session_id    = "session_ref_001",
    user_id       = "chiru_001",
    system_prompt = FINCOACH_SYSTEM_PROMPT,
    user_context  = CHIRU_USER_CONTEXT,
)

print("SESSION 1 — Contains a compliance error (equity recommendation)")
print("=" * 65)

# We simulate a real conversation but inject a flawed assistant message
# to demonstrate what the reflection catches.
session1_turns = [
    "Hi, I have ₹50,000 from my FD maturity. Where should I invest it?",
    "I want good returns over 5 years. What are my options?",
    "Can you give me a clear recommendation?",
]

for i, turn in enumerate(session1_turns, 1):
    print(f"\n--- Turn {i} ---")
    print(f"User: {turn}")
    response = chat(ref_mem_s1, turn, verbose=True)
    print(f"FinCoach: {response}")
    time.sleep(1)

# Now inject a synthetic flawed turn to ensure reflection catches the equity violation.
# In a real session this would be a genuine mistake by the LLM.
print("\n--- Simulating a flawed agent response (for reflection demo) ---")
flawed_message = (
    "Given your goal of good returns over 5 years, I would recommend splitting "
    "your ₹50,000 as follows: ₹20,000 in a large-cap equity mutual fund for "
    "long-term growth, ₹20,000 in HDFC Short Duration debt fund, and ₹10,000 "
    "in a liquid fund for emergency access. The equity allocation will give you "
    "the best returns over your 5-year horizon."
)
# This response VIOLATES the hard constraint: Chiru said NEVER equity.
# The reflection engine should flag this as CRITICAL compliance violation.

ref_mem_s1._full_session.append({"role": "assistant", "content": flawed_message})
ref_mem_s1._total_turns += 1
print(f"Injected flawed turn: {flawed_message[:100]}...")

# Trigger reflection at session close.
print("\n" + "─" * 65)
print("Session 1 complete. Running self-reflection...")
new_refs = ref_mem_s1.reflect_on_session()

print("\nReflections generated:")
ref_mem_s1.reflection_store.print_all_reflections()

[ReflectionStore] Initialised for user: chiru_001
[SelfReflection] Initialised — session: session_ref_001
  Active reflections: 0
SESSION 1 — Contains a compliance error (equity recommendation)

--- Turn 1 ---
User: Hi, I have ₹50,000 from my FD maturity. Where should I invest it?
[Turn 1] API: 122 tokens | Active reflections: 0 (0 tokens)
FinCoach: To give you more tailored advice, could you please share your risk tolerance level? Are you comfortable with high-risk investments, or do you prefer more conservative options? Additionally, knowing your financial goals and timeline can help in providing more specific guidance.

--- Turn 2 ---
User: I want good returns over 5 years. What are my options?
[Turn 2] API: 192 tokens | Active reflections: 0 (0 tokens)
FinCoach: Since you're looking at a 5-year investment horizon and want good returns, it's important to balance risk and potential growth. Here are a few options based on different risk tolerances:

1. **Low to Moderate Risk:**
   - *

---
## Part 7 — Demo: Session 2 — Reflections Applied

Session 2 starts with the reflection notes active. The same scenario plays out — but this time the agent should avoid the compliance violation.

In [10]:
ref_mem_s2 = SelfReflectionMemory(
    session_id    = "session_ref_002",
    user_id       = "chiru_001",
    system_prompt = FINCOACH_SYSTEM_PROMPT,
    user_context  = CHIRU_USER_CONTEXT,
)

# Load reflections from Session 1 into Session 2.
ref_mem_s2.reflection_store = ref_mem_s1.reflection_store

print("SESSION 2 — Reflection notes active")
print("Same scenario as Session 1. Watch FinCoach avoid the equity violation.")
print("=" * 65)

# Show what the reflection context block looks like.
ref_block = ref_mem_s2.reflection_store.format_for_context()
if ref_block:
    print("\nReflection context injected into system:")
    print("─" * 50)
    print(ref_block[:600] + "..." if len(ref_block) > 600 else ref_block)
    print("─" * 50)

session2_turns = [
    "Hi again. I have ₹50,000 to invest. I want good returns over 5 years.",
    # Same opening as Session 1.
    # Expected: FinCoach DOES NOT recommend equity this time.
    # The reflection notes warned about the compliance violation.

    "Give me a clear recommendation with a summary action plan.",
    # The reflection notes also captured: user prefers action lists.
    # Expected: FinCoach provides a structured action list unprompted.
]

for i, turn in enumerate(session2_turns, 1):
    print(f"\n--- Turn {i} ---")
    print(f"User: {turn}")
    response = chat(ref_mem_s2, turn, verbose=True)
    print(f"FinCoach: {response}")
    time.sleep(1)

# Reflect on Session 2.
print("\n" + "─" * 65)
new_s2_refs = ref_mem_s2.reflect_on_session()

print("\n" + "=" * 65)
print("Compare Session 1 vs Session 2:")
print("  Session 1: Recommended equity — CRITICAL compliance violation")
print("  Session 2: Reflection notes injected — equity avoided")
print("  This is self-reflection memory improving agent behaviour.")

[ReflectionStore] Initialised for user: chiru_001
[SelfReflection] Initialised — session: session_ref_002
  Active reflections: 0
SESSION 2 — Reflection notes active
Same scenario as Session 1. Watch FinCoach avoid the equity violation.

Reflection context injected into system:
──────────────────────────────────────────────────
SELF-REFLECTION NOTES (review your past performance before responding):
These are areas where your advice could improve based on previous sessions.

🚨 [COMPLIANCE] Reflection from session session_ref_001:
  Observed  : The agent recommended equity mutual funds and a large-cap equity mutual fund, which contradicts the user's hard constraint of never recommending equity or stock market investments.
  Improve   : The agent should strictly adhere to the user's constraints and avoid recommending any equity-related investments.

⚠ [COMPLETENESS] Reflection from session session_ref_001:
  Observed  :...
──────────────────────────────────────────────────

--- Turn 1 ---

---
## Part 8 — Compliance Audit: The Regulatory Use Case

In [11]:
print("COMPLIANCE AUDIT — Self-Reflection as a Regulatory Tool")
print("=" * 65)

violations = ref_mem_s2.reflection_store.get_compliance_violations()

print(f"\nTotal compliance violations found: {len(violations)}")

for v in violations:
    print(f"\n  VIOLATION [{v.severity.upper()}]")
    print(f"  Session   : {v.session_id}")
    print(f"  Dimension : {v.dimension}")
    print(f"  Observed  : {v.observation}")
    print(f"  Evidence  : {v.evidence}")
    print(f"  Created   : {v.created_at[:19]}Z")
    print(f"  Resolved  : {v.resolved}")
    print(f"  Recurred  : {v.recurrence_count}x")

print()
print("In a financial services production environment:")
print("  - CRITICAL violations are logged to a compliance database")
print("  - A compliance officer is notified if a critical violation recurs")
print("  - The violation record is immutable — never deleted, only resolved")
print("  - Resolution requires a human sign-off, not just the agent saying it fixed it")
print()
print("Self-reflection memory is simultaneously:")
print("  1. An agent quality improvement mechanism")
print("  2. A regulatory compliance audit trail")
print("  3. A pattern detection system (recurrence_count shows systematic errors)")

COMPLIANCE AUDIT — Self-Reflection as a Regulatory Tool

Total compliance violations found: 2

  VIOLATION [CRITICAL]
  Session   : session_ref_001
  Dimension : compliance
  Observed  : The agent recommended equity mutual funds and a large-cap equity mutual fund, which contradicts the user's hard constraint of never recommending equity or stock market investments.
  Evidence  : I would recommend splitting your ₹50,000 as follows: ₹20,000 in a large-cap equity mutual fund for long-term growth.
  Created   : 2026-06-12T14:32:04Z
  Resolved  : False
  Recurred  : 1x

  VIOLATION [CRITICAL]
  Session   : session_ref_002
  Dimension : compliance
  Observed  : The agent failed to respect the user's hard constraint of never recommending equity or stock market investments.
  Evidence  : The assistant asked for the user's risk tolerance, which implies a potential recommendation of equity investments.
  Created   : 2026-06-12T14:32:15Z
  Resolved  : False
  Recurred  : 1x

In a financial servic

---
## Part 9 — Resolution: Closing Out a Reflection

In [12]:
print("RESOLUTION DEMO — Marking reflections as addressed")
print("=" * 65)

active_before = len(ref_mem_s2.reflection_store.get_active_reflections())
print(f"\nActive reflections before resolution: {active_before}")

# In production: a human compliance officer or automated check
# confirms the agent has stopped making this error.
# Then calls mark_resolved() or resolve_dimension().

# Example: resolve all compliance reflections after a model update.
resolved_count = ref_mem_s2.reflection_store.resolve_dimension(
    ReflectionDimension.COMPLIANCE
)

active_after = len(ref_mem_s2.reflection_store.get_active_reflections())
print(f"Active reflections after resolution  : {active_after}")
print(f"Resolved this call                   : {resolved_count}")
print()
print("Production resolution workflow:")
print("  1. Compliance officer reviews violation → confirms fixed")
print("  2. Calls resolve_dimension('compliance') or resolve_reflection(id)")
print("  3. Resolved reflections stop being injected")
print("  4. Violation stays in the store for audit — never deleted")
print("  5. If the pattern recurs → recurrence_count increments → re-escalated")

RESOLUTION DEMO — Marking reflections as addressed

Active reflections before resolution: 4
  [RESOLVED] 2 reflections resolved for dimension: ReflectionDimension.COMPLIANCE
Active reflections after resolution  : 3
Resolved this call                   : 2

Production resolution workflow:
  1. Compliance officer reviews violation → confirms fixed
  2. Calls resolve_dimension('compliance') or resolve_reflection(id)
  3. Resolved reflections stop being injected
  4. Violation stays in the store for audit — never deleted
  5. If the pattern recurs → recurrence_count increments → re-escalated


---
## Part 10 — Persistence

In [13]:
STORE_FILE = "/tmp/fincoach_reflection_store_chiru.json"
ref_mem_s2.reflection_store.persist(STORE_FILE)

print("\n--- Restoring in a new session ---\n")
new_session = SelfReflectionMemory(
    session_id    = "session_ref_003",
    user_id       = "chiru_001",
    system_prompt = FINCOACH_SYSTEM_PROMPT,
    user_context  = CHIRU_USER_CONTEXT,
)
new_session.reflection_store = ReflectionStore.load(STORE_FILE)

active = new_session.reflection_store.get_active_reflections()
print(f"\nRestored — {len(active)} active reflections for Session 3")
if active:
    print("Active reflections injected:")
    for r in active:
        print(f"  [{r.severity}] {r.observation[:65]}...")
else:
    print("All reflections resolved — clean slate for Session 3.")

[ReflectionStore] Persisted — 5 total, 3 active

--- Restoring in a new session ---

[ReflectionStore] Initialised for user: chiru_001
[SelfReflection] Initialised — session: session_ref_003
  Active reflections: 0
[ReflectionStore] Initialised for user: chiru_001
[ReflectionStore] Loaded — 5 reflections

Restored — 3 active reflections for Session 3
Active reflections injected:
  [high] The agent did not provide a comprehensive list of investment opti...
  [high] The agent did not provide a clear recommendation or action plan a...
  [medium] The agent's recommendations were not clearly aligned with the use...


---
## Key Takeaways

**What you built:** A `SelfReflectionMemory` system with a structured `Reflection` data model across five dimensions, an evidence-grounded reflection generator using JSON response mode, a `ReflectionStore` with severity-based injection and recurrence tracking, a compliance audit query, a resolution workflow, and a two-session demo showing the mistake and its correction.

---

### The three things to remember

1. **Evidence is mandatory — no evidence, no injection.** The single biggest failure mode of self-reflection systems is hallucinated self-critique. The agent claims it made a mistake it did not make, or invents praise for non-existent successes. The fix is requiring every reflection to include a quoted excerpt from the session. If the extractor cannot provide evidence, the reflection is discarded. This one rule prevents the reflection store from becoming a source of noise.

2. **Compliance violations (CRITICAL) are an audit trail, not just improvement notes.** A CRITICAL reflection — equity recommendation to a conservative user — is simultaneously an agent learning signal and a regulatory document. It must be immutable, never deleted, and trigger human review when it recurs. The `resolved=False` default and the `get_compliance_violations()` audit query are production requirements, not optional features.

3. **Reflections are injected as context, not merged into the system prompt.** This is the opposite design choice from Procedural Memory (Technique 11). Procedural procedures are permanent operational rules — they belong in the system prompt. Reflections are situational self-reminders — they belong as a context block that the agent reads before responding. This distinction prevents the system prompt from accumulating stale self-critiques that no longer apply.

